In [1]:
import time
import subprocess
import sys

## `csvMSG.py` – Filtering CMMD clinical data based on available folders

**Algorithm description**

- Scans `data/raw` to get the list of patient folders (IDs).
- Reads `CMMD_clinicaldata_revision.xlsx` into a pandas DataFrame.
- Filters the DataFrame, keeping only rows where `ID1` matches a folder name.
- Saves the filtered DataFrame to
  `results/csv/CMMD_clinicaldata_revision_filtered.csv`.

**Time complexity**

Let:

- \(K\): number of patient folders,
- \(N\): number of rows in the Excel file,
- \(M\): number of columns.

Steps:

- Build the set of valid IDs: \(O(K)\).
- Read the Excel file: \(O(N \cdot M)\).
- Filter rows using membership in a set: \(O(N)\).

Total:

\[
T = O(N $\cdot$ M)
\]

**Space complexity**

- DataFrame in memory: \(O(N $\cdot$ M)\).
- Set of folder names: \(O(K)\).


In [2]:
script_path = "csvMSG.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

Tiempo total de ejecución: 0.651752 segundos


## `csv_radiomics_MS_Birads.py` – Merging radiomics, molecular subtype and BI-RADS

**Algorithm description**

- Reads three CSV files from `results/csv`:
  - `radiomics_features.csv` (radiomic features),
  - `CMMD_clinicaldata_revision_clean.csv` (clinical + subtype),
  - `TOMPEI-CMMD_imaging_diagnosis_details.csv` (diagnosis + BI-RADS).
- Normalizes ID columns (e.g. `str.strip().upper()`).
- Performs pandas `merge` operations to combine:
  - radiomic features,
  - CMMD clinical/subtype data,
  - TOMPEI diagnosis and BI-RADS.
- Creates a unified `ID` column, drops redundant ID columns and reorders
  the main identifier columns to appear first.
- Saves the final merged table to `radiomics_merged.csv`.

**Time complexity**

Let:

- \(N_1, M_1\): rows and columns in `radiomics_features.csv`,
- \(N_2, M_2\): rows and columns in `CMMD_clinicaldata_revision_clean.csv`,
- \(N_3, M_3\): rows and columns in `TOMPEI-CMMD_imaging_diagnosis_details.csv`.

Steps:

- Read each CSV into a DataFrame: \(O(N_i $\cdot$ M_i)\).
- Normalize IDs: \(O(N_i)\) per DataFrame.
- Merge operations in pandas:
  - Typically implemented as hash joins, roughly linear in the combined
    number of rows.

Therefore, in aggregate:

\[
T = O(N_1 $\cdot$ M_1 + N_2 $\cdot$ M_2 + N_3 $\cdot$ M_3)
\]

**Space complexity**

- Each DataFrame consumes \(O(N_i $\cdot$ M_i)\) memory.
- The final merged DataFrame has \(N\) rows and \(M\) columns:

\[
S = O(N_1 $\cdot$ M_1 + N_2 $\cdot$ M_2 + N_3 $\cdot$ M_3)
\]


In [3]:
script_path = "csv_radiomics_MS_Birads.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

CSV generado en: /mnt/Datos/Scientific-programming-ITM-assignments/results/csv/radiomics_merged.csv
Tiempo total de ejecución: 0.599030 segundos


## `csvClinicaldata.py` – Filtering TOMPEI-CMMD clinical sheets

**Algorithm description**

- Lists patient folders available in `data/raw` to obtain valid IDs.
- Reads an Excel file (e.g. `TOMPEI-CMMD_clinical_data_v01_20250121.xlsx`) with:
  - An imaging diagnosis sheet,
  - A lesion details sheet.
- Filters rows in each sheet so that only records whose ID matches an existing
  patient folder are kept.
- Writes the filtered tables as CSV files in `results/csv`.

**Time complexity**

Let:

- \(K\): number of patient folders (IDs on disk),
- \(N\): number of rows in an Excel sheet,
- \(M\): number of columns.

Steps:

- Build the set of folder names: \(O(K)\).
- Read the Excel sheet into a pandas DataFrame: \(O(N \cdot M)\).
- Filter rows using `isin` against a set (average \(O(1)\) lookup per row):
  \(O(N)\).

Overall, dominated by reading and scanning the table:

\[
T = O(N $\cdot$ M)
\]

**Space complexity**

- The DataFrame has \(N\) rows and \(M\) columns:

\[
S = O(N $\cdot$ M)
\]

- The set of folder names uses \(O(K)\) additional space.


In [4]:
script_path = "csvClinicaldata.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

Filtered CSV files generated:
 - /mnt/Datos/Scientific-programming-ITM-assignments/results/csv/TOMPEI-CMMD_imaging_diagnosis_details.csv
 - /mnt/Datos/Scientific-programming-ITM-assignments/results/csv/TOMPEI-CMMD_lesion_details.csv
Tiempo total de ejecución: 1.354227 segundos


## `Molecularsubtypeclean.py` – Cleaning molecular subtype and copying patient folders

**Algorithm description**

- Reads `CMMD_clinicaldata_revision.xlsx` into a pandas DataFrame.
- Filters out rows where:
  - `classification == 'Malignant'` and
  - `subtype` is missing (NaN).
- For each remaining row:
  - Builds the path to the patient folder in `data/raw`.
  - Copies the entire patient folder to `data/raw csv` using `shutil.copytree`.

**Time complexity**

Let:

- \(N\): number of rows in the original Excel sheet,
- \(M\): number of columns,
- \(N'\): number of rows after filtering.

Steps:

- Read and filter the DataFrame: \(O(N $\cdot$ M)\).
- For each of the \(N'\) selected patients:
  - Copy the folder recursively. The cost depends on the total file size
    per folder; if we assume a bounded average size, we can model it as
    \(O(1)\) per patient.

Under that assumption:

\[
T = O(N $\cdot$ M + N')
\]

In practice, if folders are large, the real runtime will be dominated by
disk I/O.

**Space complexity**

- The DataFrame uses \(O(N $\cdot$ M)\) memory.
- Folder copies are written to disk, not held in RAM.

\[
S = O(N $\cdot$ M)
\]


In [5]:
script_path = "Molecularsubtypeclean.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

Tiempo total de ejecución: 6.862609 segundos
